### PydanticOutputParser

In [1]:
!pip --version

pip 25.0.1 from D:\hanhwa0902\ex0916\.0916venv\Lib\site-packages\pip (python 3.12)



In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_teddynote import logging

logging.langsmith("test0916")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0916


In [6]:
from langchain_teddynote.messages import stream_response

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

llm = ChatOpenAI(temperature=0, model_name="gpt-5-mini")

In [8]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [ ]:
# 출력 파서를 사용하지 않는 경우

from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0, model_name="gpt-5-mini")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

중요 내용 요약(간결하게):

- 발신자: 김철수 상무이사, 바이크코퍼레이션 (chulsoo.kim@bikecorporation.me)  
- 수신자: 이은채 대리 (eunchae@teddyinternational.me)  
- 제목: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

- 목적: 귀사 신제품 ZENESIS의 유통 협력 가능성 논의
  - 바이크코퍼레이션은 자전거 제조·유통 분야의 경험과 전문성 보유

- 요청사항: ZENESIS의 상세 브로슈어 제출 요청 (특히)
  - 기술 사양
  - 배터리 성능
  - 디자인 정보
  → 자료를 받아 유통 전략 및 마케팅 계획 구체화 예정

- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 회의 제안

- 마무리: 회신 및 자료 제출·미팅 확정 요청 implied

원하시면 수신자 입장에서의 회신 예시(간단)도 만들어 드리겠습니다.

In [10]:
print(output)

중요 내용 요약(간결하게):

- 발신자: 김철수 상무이사, 바이크코퍼레이션 (chulsoo.kim@bikecorporation.me)  
- 수신자: 이은채 대리 (eunchae@teddyinternational.me)  
- 제목: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

- 목적: 귀사 신제품 ZENESIS의 유통 협력 가능성 논의
  - 바이크코퍼레이션은 자전거 제조·유통 분야의 경험과 전문성 보유

- 요청사항: ZENESIS의 상세 브로슈어 제출 요청 (특히)
  - 기술 사양
  - 배터리 성능
  - 디자인 정보
  → 자료를 받아 유통 전략 및 마케팅 계획 구체화 예정

- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 회의 제안

- 마무리: 회신 및 자료 제출·미팅 확정 요청 implied

원하시면 수신자 입장에서의 회신 예시(간단)도 만들어 드리겠습니다.


In [ ]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

# PydanticOutputParser 생성
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [ ]:
# PydanticOutputParser 매서드 1: get_format_instructions(): 언어 모델이 출력해야 할 정보의 형식을 정의하는 지침을 제공
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [13]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

prompt = prompt.partial(format=parser.get_format_instructions())

In [14]:
chain = prompt | llm

In [16]:
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

output = stream_response(response, return_output=True)

{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션 김철수 상무가 상대 회사의 신규 자전거 ZENESIS에 관심을 표명하며, 유통 협력 가능성을 논의하고자 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 제공을 요청함. 이를 바탕으로 유통 전략 및 마케팅 계획을 구체화하려 하며, 협력 논의를 위한 미팅을 제안함.",
  "date": "다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 미팅 제안"
}

In [18]:
# PydanticOutputParser 핵심 메서드2: parse(): 언어 모델의 출력을 받아 이를 특정 구조로 분석하고 변환
structured_output = parser.parse(output)
print(structured_output)

structured_output.person

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 상대 회사의 신규 자전거 ZENESIS에 관심을 표명하며, 유통 협력 가능성을 논의하고자 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 제공을 요청함. 이를 바탕으로 유통 전략 및 마케팅 계획을 구체화하려 하며, 협력 논의를 위한 미팅을 제안함.' date='다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 미팅 제안'


'김철수'

In [19]:
# parser가 추가된 체인 생성
chain = prompt | llm | parser

In [20]:
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션의 김철수가 테디인터내셔널의 ZENESIS 자전거에 대해 관심을 표하며, 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 제공을 요청함. 이를 바탕으로 자사 유통 전략 및 마케팅 계획을 구체화하고자 하며, 협력 가능성 논의를 위해 미팅을 제안함. 회사는 자전거 제조·유통 분야의 경험과 전문성을 보유하고 있음을 밝힘.', date='1월 15일(화요일) 오전 10시')

### with_structured_output()

In [22]:
# .with structured_output(Pydantic)을 사용하여 출력 파서를 추가하면, 출력을 Pydantic 객체로 변환 가능

llm_with_structered = ChatOpenAI(
    temperature=0, model_name="gpt-5-mini"
).with_structured_output(EmailSummary)

In [25]:
answer = llm_with_structered.invoke(email_conversation)
answer

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션 김철수 상무가 ZENESIS 모델의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 제공을 요청하고, 유통 협력 및 마케팅 논의를 위해 1월 15일(화) 오전 10시에 귀사 사무실에서 미팅을 제안함.', date='1월 15일(화) 오전 10시')

In [26]:
answer.email

'chulsoo.kim@bikecorporation.me'